# BSG Tarea 07 - Sistemas de Gestión de Bases de Datos con Python 

### Pregunta 1. Diferencias clave entre bases de datos NoSQL y relacionales

**Modelo de datos:**  
Las bases de datos relacionales almacenan la información en tablas con un esquema fijo y relaciones entre ellas (llaves primarias y foráneas).  
En cambio, las bases de datos NoSQL utilizan modelos de datos flexibles: **documentos** (MongoDB), **clave-valor** (Redis), **columna ancha** (Cassandra) o **grafo** (Neo4j).  
Esto permite manejar estructuras variables sin necesidad de alterar el esquema.

**Escalamiento:**  
Las bases de datos relacionales suelen escalar de forma *vertical*, aumentando la capacidad de un único servidor.  
Las NoSQL están diseñadas para escalar de forma *horizontal*, distribuyendo los datos entre múltiples nodos mediante técnicas de particionado (*sharding*).

**Consistencia y disponibilidad (teorema CAP):**  
Las bases de datos relacionales priorizan la **consistencia fuerte** y cumplen el modelo ACID.  
Las NoSQL suelen ofrecer mayor **disponibilidad** y **tolerancia a particiones**, sacrificando en algunos casos la consistencia inmediata a favor de una *consistencia eventual* configurable.

**Consultas:**  
Las bases relacionales utilizan el lenguaje **SQL**, que permite realizar consultas complejas con *JOINs* y agregaciones.  
En cambio, las NoSQL emplean **APIs propias** o lenguajes de consulta optimizados para operaciones rápidas sobre estructuras simples, sin requerir *joins* entre múltiples colecciones o tablas.

**Casos de uso típicos:**  
Las bases relacionales son adecuadas para sistemas donde la integridad de los datos es crítica, como aplicaciones financieras, ERP o CRM.  
Las NoSQL son preferibles cuando se manejan grandes volúmenes de información, estructuras cambiantes y necesidad de baja latencia, como en aplicaciones web, redes sociales, catálogos o sistemas de registro de eventos.

**Ejemplo de escenario donde NoSQL es preferible:**  
Una aplicación móvil global que registra millones de eventos por minuto y muestra contenido personalizado para cada usuario.  
*MongoDB* sería una opción adecuada para almacenar perfiles y contenidos en documentos con atributos dinámicos.  
Por otro lado, *Cassandra* resulta ideal para escrituras masivas y lectura distribuida con alta disponibilidad entre múltiples centros de datos.

### Pregunta 2. Python + MongoDB (local): crear `mi_base` y colección `usuarios`

In [6]:
from pymongo import MongoClient
from pymongo.errors import ServerSelectionTimeoutError
from pprint import pprint

# Conexión al servidor local
URI = "mongodb://127.0.0.1:27017"   # localhost por defecto
client = MongoClient(URI, serverSelectionTimeoutMS=3000)

try:
    # ping rápido para validar conexión
    client.admin.command("ping")
    print("✅ Conectado a MongoDB local.")
except ServerSelectionTimeoutError as e:
    raise SystemExit(f"❌ No se pudo conectar a MongoDB en {URI}. "
                     f"¿Está corriendo el servicio?\nDetalle: {e}")


✅ Conectado a MongoDB local.


In [7]:
# Crear/seleccionar DB y colección
db = client["mi_base"]
usuarios = db["usuarios"]

print("DB y colección listas:", db.name, usuarios.name)

DB y colección listas: mi_base usuarios


In [8]:
# Insertar 3 documentos semiestructurados
docs = [
    {
        "nombre": "Ana",
        "edad": 28,
        "hobbies": ["fotografía", "senderismo"]
    },
    {
        "nombre": "Luis",
        "edad": 35,
        "hobbies": ["ajedrez", "running", "cocina"]
    },
    {
        "nombre": "María",
        "edad": 22,
        "hobbies": ["lectura"]
    }
]

res = usuarios.insert_many(docs)

In [9]:
# Verificar: contar y listar (bonito)
total = usuarios.count_documents({})
print(f"Total documentos en 'usuarios': {total}\n")

for doc in usuarios.find({}, {"_id": 0}).limit(10):
    pprint(doc)

Total documentos en 'usuarios': 6

{'edad': 28, 'hobbies': ['fotografía', 'senderismo'], 'nombre': 'Ana'}
{'edad': 35, 'hobbies': ['ajedrez', 'running', 'cocina'], 'nombre': 'Luis'}
{'edad': 22, 'hobbies': ['lectura'], 'nombre': 'María'}
{'edad': 28, 'hobbies': ['fotografía', 'senderismo'], 'nombre': 'Ana'}
{'edad': 35, 'hobbies': ['ajedrez', 'running', 'cocina'], 'nombre': 'Luis'}
{'edad': 22, 'hobbies': ['lectura'], 'nombre': 'María'}


### Pregunta 3. ¿Cómo funciona el modelo de datos distribuido en Cassandra y por qué es adecuado para alta disponibilidad?

**Arquitectura (peer-to-peer)**
- Todos los nodos son **iguales**; no hay maestro. Forman un anillo y se comunican por **Gossip** (descubren estado y pertenencia).
- Soporta **múltiples centros de datos (multi-DC)** con conocimiento de topología (“snitches”).

**Particionado y réplica**
- Cada fila se ubica por una **clave de partición**; un hash consistente determina el **nodo(s)** que la almacenan.
- El **Replication Factor (RF)** define cuántas copias hay por **keyspace** (p. ej., RF=3 → 3 réplicas).
- Con **vNodes**, cada nodo posee muchos **rangos** del anillo ⇒ re-balanceo y escalado horizontal más simples.

**Consistencia ajustable (tunable)**
- Puedes elegir por operación: `ONE`, `QUORUM`, `LOCAL_QUORUM`, `ALL`, etc.  
- Con RF=3, usar `QUORUM` (2/3) en lecturas y/o escrituras da **consistencia fuerte** sin sacrificar toda la disponibilidad.
- `ONE` favorece baja latencia y disponibilidad, admitiendo **consistencia eventual**.

**¿Por qué es adecuado para alta disponibilidad?**
- Diseñado para el **AP** del teorema CAP (Alta **Disponibilidad** y tolerancia a **Particiones**), con consistencia ajustable por operación.
- **Multi-región/ multi-DC** nativo para aplicaciones 24/7 globales.
- Escrituras **siempre disponibles** (append + LSM), latencias predecibles y throughput elevado.

**Trade-offs clave**
- No hay JOINs/aggregations complejos como en un RDBMS; se prefiere **duplicar datos** por consulta.
- Requiere **modelado cuidadoso de la clave de partición** para evitar *hotspots* y garantizar distribución uniforme.

> En resumen: Cassandra distribuye y replica datos por clave de partición en un anillo peer-to-peer, con consistencia ajustable y mecanismos de reparación; esto, junto con su diseño LSM y multi-DC, le permite ofrecer **baja latencia y alta disponibilidad** a gran escala.


### Pregunta 4. MongoDB + PyMongo: filtrar `edad > 30` y agrupar por `hobbies` (conteo)

In [10]:
from pymongo import MongoClient
from pprint import pprint

# Conexión
client = MongoClient("mongodb://127.0.0.1:27017", serverSelectionTimeoutMS=3000)
db = client["mi_base"]
usuarios = db["usuarios"]

# Índices
usuarios.create_index("edad")

# Pipeline de agregación
pipeline = [
    {"$match": {"edad": {"$gt": 30}}},
    {"$unwind": "$hobbies"},
    {"$group": {"_id": "$hobbies", "conteo": {"$sum": 1}}},
    {"$sort": {"conteo": -1, "_id": 1}}
]

resultado = list(usuarios.aggregate(pipeline))
print("Resultado (hobby -> conteo) para usuarios con edad > 30:\n")
for doc in resultado:
    print(f"{doc['_id']}: {doc['conteo']}")

Resultado (hobby -> conteo) para usuarios con edad > 30:

ajedrez: 2
cocina: 2
running: 2
